# Longitudinal Modulus from Compression: Final Equilibrated State

## Theory

$$M = \frac{\sigma'_{zz}}{\varepsilon_{zz}}$$

Network stress from poroelasticity decomposition:
$$p_p(z) = -\frac{\sigma_{s,zz}(z)}{\phi_s(z)}, \qquad \sigma'_{zz}(z) = \sigma_{p,zz}(z) + \phi_p(z)\,p_p(z)$$

This notebook:
- Plots time evolution of φ, p_p, σ' to verify equilibration
- Uses **final timestep only** for M calculation
- Reports gel-averaged M with 95% CI from spatial variation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
from pathlib import Path
print('Imports successful')

## Configuration

In [ ]:
sim_name = 'walled_slab_support_tall_angle_8_1.0_1.0_10000000_50000'
stress_data_dir = '../../flow_data_local/partial_stress_data/'
traj_file = f'../../flow_data_local/traj_files/gel_flow_{sim_name}.lammpstrj'
output_folder = '../../flow_data_local/flow_plots/'
binWidth = 2.0
phi_gel_threshold = 0.05
ci_level = 0.95

## Helper Functions

In [ ]:
def read_ave_time_file(filepath):
    data_by_time = []
    with open(filepath, 'r') as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            timestep, nrows = int(parts[0]), int(parts[1])
            values = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2:
                        values.append(float(vp[1]))
            if values:
                data_by_time.append((timestep, np.arange(1, len(values)+1), np.array(values)))
            i += nrows + 1
        else:
            i += 1
    return data_by_time


def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    if len(rows) == 2 and all(len(r)==1 for r in rows):
        return None, np.array([rows[0][0]]), np.array([rows[1][0]])
    if len(rows) == 1 and len(rows[0]) == 2:
        return None, np.array([rows[0][0]]), np.array([rows[0][1]])
    if all(len(r)==3 for r in rows):
        arr = np.array(rows)
        ts, col1, col2 = arr[:,0].astype(int).tolist(), arr[:,1], arr[:,2]
        ratio = np.nanmedian(col2) / np.nanmedian(col1)
        dL = col2 if ratio < 0.5 else col1 - col2
        return ts, col1, dL
    raise ValueError(f'Unrecognised format in {filepath}')


def read_lammpstrj_frame(filepath, frame_idx=0):
    with open(filepath) as f:
        lines = f.readlines()
    starts = [i for i,l in enumerate(lines) if 'ITEM: TIMESTEP' in l]
    if frame_idx < 0: frame_idx = len(starts) + frame_idx
    s = starts[frame_idx]
    e = starts[frame_idx+1] if frame_idx+1<len(starts) else len(lines)
    fl = lines[s:e]
    timestep = int(fl[1].strip())
    box = {'x': [float(v) for v in fl[5].split()],
           'y': [float(v) for v in fl[6].split()],
           'z': [float(v) for v in fl[7].split()]}
    atoms = [[int(p[0]),int(p[1]),int(p[2]),float(p[3]),float(p[4]),float(p[5])]
             for p in [l.split() for l in fl[9:9+int(fl[3].strip())]]]
    return timestep, box, atoms


def compute_volume_fractions_1d(atoms_data, box_bounds, bin_width, direction='z'):
    a = np.array(atoms_data)
    typ, pos = a[:,1].astype(int), a[:,3:6]
    pm, sm = (typ==1)|(typ==2), (typ==3)
    d = {'x':0,'y':1,'z':2}[direction]
    lo, hi = box_bounds[direction]
    edges = np.arange(lo, hi+bin_width, bin_width)
    centers = (edges[:-1]+edges[1:])/2
    phi_p, phi_s = np.zeros(len(centers)), np.zeros(len(centers))
    for i,(a_,b_) in enumerate(zip(edges[:-1],edges[1:])):
        mask = (pos[:,d]>=a_)&(pos[:,d]<b_)
        np_, ns_, nt = np.sum(pm&mask), np.sum(sm&mask), np.sum(mask)
        if nt>0: phi_p[i], phi_s[i] = np_/nt, ns_/nt
    return centers, phi_p, phi_s


def mean_ci(values, ci_level=0.95):
    v = values[~np.isnan(values)]
    n = len(v)
    if n==0: return np.nan, np.nan, np.nan
    if n==1: return v[0], v[0], v[0]
    m = np.mean(v)
    lo, hi = stats.t.interval(ci_level, df=n-1, loc=m, scale=stats.sem(v))
    return m, lo, hi


print('Helper functions defined')

## Step 1: Read All Data

In [ ]:
stress_data = {}
for key, fp in [
    ('polymer_z', f'{stress_data_dir}stress_z_polymer_{sim_name}.dat'),
    ('solvent_z', f'{stress_data_dir}stress_z_solvent_{sim_name}.dat'),
]:
    print(f'Reading {key} ...')
    stress_data[key] = read_ave_time_file(fp)

n_stress = len(stress_data['polymer_z'])
print(f'\nStress snapshots: {n_stress}')

strain_file = f'{stress_data_dir}strain_zz_{sim_name}.dat'
print(f'\nReading {strain_file}')
ts_strain, L_arr, dL_arr = read_strain_file(strain_file)

if len(L_arr)==1:
    L_final, dL_final = L_arr[0], dL_arr[0]
else:
    L_final, dL_final = L_arr[-1], dL_arr[-1]

eps_final = dL_final / L_final
print(f'\nFinal: L={L_final:.4f}, ΔL={dL_final:.4f}, ε={eps_final:.4f}')

## Step 2: Process All Timesteps (for equilibration plots)

In [ ]:
print(f'Reading {traj_file}')
with open(traj_file) as f:
    n_frames = sum(1 for l in f if 'ITEM: TIMESTEP' in l)
print(f'  {n_frames} frames')

n_snapshots = min(n_frames, n_stress)
print(f'Processing {n_snapshots} snapshots ...\n')

all_timesteps = []
all_phi_polymer, all_phi_solvent = [], []
all_p_p = []
all_sigma_p_zz, all_sigma_s_zz = [], []
all_sigma_prime_zz = []

for idx in range(n_snapshots):
    if (idx+1) % max(1,n_snapshots//10)==0 or idx==0:
        print(f'  {idx+1}/{n_snapshots}')

    _, box, atoms = read_lammpstrj_frame(traj_file, idx)
    z_centers, phi_p, phi_s = compute_volume_fractions_1d(atoms, box, binWidth, 'z')

    ts = stress_data['polymer_z'][idx][0]
    bins_z = stress_data['polymer_z'][idx][1]
    z_coords = bins_z * binWidth - binWidth/2
    sig_p = stress_data['polymer_z'][idx][2]
    sig_s = stress_data['solvent_z'][idx][2]

    phi_s_safe = np.where(phi_s>1e-6, phi_s, np.nan)
    p_p = -sig_s / phi_s_safe
    sig_prime = sig_p + phi_p * np.where(np.isnan(p_p), 0.0, p_p)

    all_timesteps.append(ts)
    all_phi_polymer.append(phi_p)
    all_phi_solvent.append(phi_s)
    all_p_p.append(p_p)
    all_sigma_p_zz.append(sig_p)
    all_sigma_s_zz.append(sig_s)
    all_sigma_prime_zz.append(sig_prime)

print(f'\nDone. t = {all_timesteps[0]} → {all_timesteps[-1]}')

zlo, zhi = box['z']
Lz = zhi - zlo
z_norm = (z_coords - zlo) / Lz
z_centers_norm = (z_centers - zlo) / Lz

## Step 3: Equilibration Check — Time Evolution Plots

Verify φ, p_p, and σ' have reached steady-state before extracting M.

In [ ]:
os.makedirs(output_folder, exist_ok=True)
colors = plt.cm.viridis(np.linspace(0,1,n_snapshots))
alpha = 0.35 if n_snapshots>5 else 0.75

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f'Equilibration Check: {sim_name}\n{n_snapshots} snapshots',
    fontsize=12, fontweight='bold')

from matplotlib.lines import Line2D
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import Normalize

# (a) Volume fractions
ax = axes[0]
for i in range(n_snapshots):
    ax.plot(z_centers_norm, all_phi_polymer[i], '-',  color=colors[i], lw=1.5, alpha=alpha)
    ax.plot(z_centers_norm, all_phi_solvent[i], '--', color=colors[i], lw=1.5, alpha=alpha)
ax.legend(handles=[
    Line2D([0],[0], color='gray', lw=2,       label=r'$\phi_p$'),
    Line2D([0],[0], color='gray', lw=2, ls='--', label=r'$\phi_s$'),
], fontsize=9)
ax.set(xlabel='z/Lz', ylabel='Volume Fraction', title='(a) φ(z,t)', xlim=(0,1))
ax.grid(alpha=0.3)

# (b) Pore pressure
ax = axes[1]
for i in range(n_snapshots):
    ax.plot(z_norm, all_p_p[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='z/Lz', ylabel=r'$p_p$', title='(b) Pore Pressure(z,t)', xlim=(0,1))
ax.grid(alpha=0.3)

# (c) Network stress
ax = axes[2]
for i in range(n_snapshots):
    ax.plot(z_norm, all_sigma_prime_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='z/Lz', ylabel=r"$\sigma'_{zz}$", title=r"(c) Network Stress(z,t)", xlim=(0,1))
ax.grid(alpha=0.3)

# Colorbar
cax = fig.add_axes([0.92, 0.15, 0.015, 0.70])
cb = ColorbarBase(cax, cmap='viridis',
                  norm=Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1]))
cb.set_label('Timestep', fontsize=10)

plt.tight_layout(rect=[0,0,0.91,1])
out1 = Path(output_folder) / f'equilibration_check_{sim_name}.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
print(f'Saved: {out1}')
plt.show()

print('\nLast curves (purple → yellow = early → late) should overlap if equilibrated.')

## Step 4: Final-State Longitudinal Modulus

Using **final timestep only** to compute M(z) from equilibrated stress state.

In [ ]:
idx_final = -1

phi_p_final = all_phi_polymer[idx_final]
phi_s_final = all_phi_solvent[idx_final]
p_p_final = all_p_p[idx_final]
sig_p_final = all_sigma_p_zz[idx_final]
sig_s_final = all_sigma_s_zz[idx_final]
sig_prime_final = all_sigma_prime_zz[idx_final]

if abs(eps_final) < 1e-12:
    raise ValueError('Final strain ≈ 0 — cannot compute M')

M_final = sig_prime_final / eps_final

print(f'Final timestep: {all_timesteps[idx_final]}')
print(f'ε_zz = {eps_final:.4f}')
print(f'M(z) range: [{np.nanmin(M_final):.3f}, {np.nanmax(M_final):.3f}]')

## Step 5: Spatial Profile of M(z) at Final State

In [ ]:
fig2, axes2 = plt.subplots(1, 4, figsize=(18,5))
fig2.suptitle(
    f'Final Equilibrated State: {sim_name}\nt = {all_timesteps[idx_final]}',
    fontsize=12, fontweight='bold')

# (a) φ
ax = axes2[0]
ax.plot(z_centers_norm, phi_p_final, 'o-', label=r'$\phi_p$', color='steelblue', ms=3)
ax.plot(z_centers_norm, phi_s_final, 's--', label=r'$\phi_s$', color='coral', ms=3)
ax.legend(fontsize=9)
ax.set(xlabel='z/Lz', ylabel='Volume Fraction', title='(a) φ(z)', xlim=(0,1))
ax.grid(alpha=0.3)

# (b) p_p
ax = axes2[1]
ax.plot(z_norm, p_p_final, 'o-', color='green', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='z/Lz', ylabel=r'$p_p$', title='(b) Pore Pressure(z)', xlim=(0,1))
ax.grid(alpha=0.3)

# (c) σ'
ax = axes2[2]
ax.plot(z_norm, sig_prime_final, 'o-', color='purple', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='z/Lz', ylabel=r"$\sigma'_{zz}$", title=r"(c) Network Stress(z)", xlim=(0,1))
ax.grid(alpha=0.3)

# (d) M
ax = axes2[3]
ax.plot(z_norm, M_final, 'o-', color='crimson', ms=3)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='z/Lz', ylabel='M', title=r'(d) Modulus(z) = $\sigma\'_{zz}/\varepsilon$', xlim=(0,1))
ax.grid(alpha=0.3)

plt.tight_layout()
out2 = Path(output_folder) / f'final_state_profiles_{sim_name}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved: {out2}')
plt.show()

## Step 6: Gel-Averaged M with Confidence Interval

**Procedure:**
1. Identify gel bins: where φ_p > threshold (0.05)
2. Extract M values from those bins: $\{M_1, M_2, \dots, M_n\}$
3. Compute sample mean: $\bar{M} = \frac{1}{n}\sum M_i$
4. Compute standard error: $\text{SEM} = s / \sqrt{n}$ where $s = \sqrt{\frac{1}{n-1}\sum(M_i - \bar{M})^2}$
5. Construct 95% CI using t-distribution with $n-1$ degrees of freedom:
   $$\text{CI} = \bar{M} \pm t_{0.975, n-1} \times \text{SEM}$$

**Interpretation:** The CI reflects **spatial heterogeneity** in the gel's stiffness.
Narrow band = uniform; wide band = some bins are stiffer/softer than average.

In [ ]:
print('='*70)
print('DERIVATION OF GEL-AVERAGED LONGITUDINAL MODULUS')
print('='*70)

# Step 1: Identify gel bins
gel_mask = phi_p_final > phi_gel_threshold
n_gel_bins = np.sum(gel_mask)
print(f'\n1. Gel bins identified: {n_gel_bins} / {len(phi_p_final)} (φ_p > {phi_gel_threshold})')

# Step 2: Extract M values
M_gel = M_final[gel_mask]
M_gel = M_gel[~np.isnan(M_gel)]  # remove any NaNs
n_valid = len(M_gel)
print(f'   Valid M values: {n_valid}')
print(f'   M_gel range: [{M_gel.min():.4f}, {M_gel.max():.4f}]')

if n_valid == 0:
    raise ValueError('No valid M values in gel region')

# Step 3: Sample mean
M_mean = np.mean(M_gel)
print(f'\n2. Sample mean: M̄ = {M_mean:.6f}')

# Step 4: Standard error
if n_valid == 1:
    print('\n   WARNING: only 1 data point — CI = point estimate')
    M_ci_lo, M_ci_hi = M_mean, M_mean
else:
    s = np.std(M_gel, ddof=1)  # sample std dev
    sem = stats.sem(M_gel)     # s / sqrt(n)
    print(f'\n3. Sample std dev: s = {s:.6f}')
    print(f'   Standard error: SEM = s/√n = {sem:.6f}')

    # Step 5: t-distribution CI
    df = n_valid - 1
    t_crit = stats.t.ppf((1+ci_level)/2, df)
    margin = t_crit * sem
    M_ci_lo = M_mean - margin
    M_ci_hi = M_mean + margin

    print(f'\n4. t-distribution (df={df}, α={1-ci_level}):')
    print(f'   Critical value: t_{{0.975, {df}}} = {t_crit:.4f}')
    print(f'   Margin of error: {margin:.6f}')

ci_pct = int(ci_level*100)
print(f'\n{"="*70}')
print(f'FINAL RESULT ({ci_pct}% Confidence Interval)')
print(f'{"="*70}')
print(f'  M = {M_mean:.4f}  [{M_ci_lo:.4f}, {M_ci_hi:.4f}]')
print(f'\nStrain: ε_zz = {eps_final:.4f}')
print(f'Gel thickness: L = {L_final:.2f}, ΔL = {dL_final:.2f}')
print('='*70)

## Optional: Save Data

In [ ]:
out_data = Path(output_folder) / f'longitudinal_modulus_final_{sim_name}.npz'
np.savez(out_data,
         z_coords=z_coords, z_norm=z_norm,
         z_centers=z_centers, z_centers_norm=z_centers_norm,
         final_timestep=all_timesteps[idx_final],
         L=L_final, dL=dL_final, eps=eps_final,
         phi_p=phi_p_final, phi_s=phi_s_final,
         p_p=p_p_final, sigma_p_zz=sig_p_final, sigma_s_zz=sig_s_final,
         sigma_prime_zz=sig_prime_final, M=M_final,
         M_mean=M_mean, M_ci_lo=M_ci_lo, M_ci_hi=M_ci_hi,
         ci_level=ci_level, n_gel_bins=n_gel_bins,
         sim_name=sim_name)
print(f'Saved: {out_data}')